# Modelo TF-IDF

**Curso:** Procesamiento de Lenguaje Natural · Unidad 2 — Semántica distribucional y representación de texto
**Institución:** Universidad Tecnológica de Bolívar
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 6 — Vector Semantics (§6.5 TF-IDF)
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13
**Prerrequisito:** `bow_model.ipynb` (modelo Bag-of-Words)

---

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Explicar qué es **TF-IDF** (*Term Frequency–Inverse Document Frequency*) y por qué mejora los conteos brutos de BoW
2. **Calcular TF-IDF paso a paso** (TF → DF → IDF → TF×IDF → normalización L2) y verificar el resultado contra `TfidfVectorizer`
3. Distinguir la fórmula de IDF del libro de la que usa scikit-learn por defecto
4. Usar **n-gramas** (unigramas + bigramas) para capturar expresiones de varias palabras
5. Comparar BoW y TF-IDF sobre los mismos documentos
6. Identificar los **términos clave** de un documento y medir la **similitud coseno** entre documentos
7. Reconocer cómo los **documentos duplicados** distorsionan el IDF en un corpus real

## Estructura

| Parte | Corpus | Propósito |
|-------|--------|-----------|
| **A** (secciones 2–6) | 5 documentos de juguete | Entender cada cálculo a mano |
| **B** (secciones 7–10) | `dataset_reseñas_espanol.csv` (200 reseñas) | Aplicar TF-IDF a datos reales |


## ¿Qué es TF-IDF?

En BoW cada palabra pesa lo mismo: un artículo como *la* cuenta igual que una palabra informativa como *excelente*. TF-IDF combina dos ideas:

- **TF (frecuencia del término):** una palabra que aparece mucho en un documento probablemente es importante **para ese documento**.
- **IDF (frecuencia inversa de documento):** una palabra que aparece en **muchos documentos** del corpus no sirve para distinguirlos, así que su peso se reduce.

### Notación

| Símbolo | Significado |
|---------|-------------|
| $N$ | Número total de documentos del corpus |
| $\text{tf}(t, d)$ | Número de veces que el término $t$ aparece en el documento $d$ |
| $\text{df}(t)$ | Número de **documentos** que contienen $t$ (no cuántas veces aparece) |

### Fórmula del libro (Jurafsky & Martin, ec. 6.12–6.13)

$$\text{tf}(t,d) = \log_{10}\big(\text{count}(t,d) + 1\big) \qquad \text{idf}(t) = \log_{10}\left(\frac{N}{\text{df}(t)}\right)$$

### Fórmula de scikit-learn por defecto (`TfidfVectorizer`)

$$\text{tf}(t,d) = \text{count}(t,d) \qquad \text{idf}(t) = \ln\left(\frac{1 + N}{1 + \text{df}(t)}\right) + 1$$

y después **cada fila (documento) se normaliza a norma L2 = 1**:

$$\text{tfidf}(t,d) = \frac{\text{tf}(t,d)\cdot\text{idf}(t)}{\sqrt{\sum_{t'} \big(\text{tf}(t',d)\cdot\text{idf}(t')\big)^2}}$$

### ¿Por qué difieren?

| Detalle | Libro | scikit-learn (por defecto) | Parámetro que lo controla |
|---------|-------|----------------------------|---------------------------|
| TF | logarítmico | conteo bruto | `sublinear_tf=False` (con `True` usa $1 + \ln(\text{tf})$) |
| Logaritmo | base 10 | natural ($\ln$) | — |
| Suavizado $(1+N)/(1+\text{df})$ | no | sí: actúa como si existiera un documento extra con todos los términos y evita dividir por cero | `smooth_idf=True` |
| “+1” final en IDF | no | sí: un término presente en **todos** los documentos no queda con peso 0 | — |
| Normalización | no | L2 por documento | `norm='l2'` |

> Ninguna fórmula es “la correcta”: son variantes del mismo principio. Lo importante es saber **cuál usa tu código** para interpretar bien los números. En este notebook calculamos ambas y verificamos la de scikit-learn.


---

## 1. Configuración del entorno


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

DIR_DATA = Path.cwd() / 'data'
print(f'Directorio de datos: {DIR_DATA}')
print(f'¿Existe el directorio?: {DIR_DATA.exists()}')

---

# Parte A — Cálculo paso a paso con un corpus de juguete

## 2. Definir el corpus

Cinco reseñas muy cortas. Están escritas **sin tildes** (*pelicula*, *gusto*) a propósito, para que todas las formas coincidan. En texto real, *película* y *pelicula* serían **dos términos distintos**, salvo que se use `strip_accents='unicode'`.


In [ ]:
corpus = [
    'buena pelicula',
    'no me gusto la pelicula',
    'no me gusto',
    'me gusto la pelicula',
    'buena',
]
doc_ids = [f'd{i}' for i in range(1, len(corpus) + 1)]
N = len(corpus)

pd.DataFrame({'documento': corpus}, index=doc_ids)

### 2.1 ¿Cómo tokeniza scikit-learn?

`CountVectorizer` y `TfidfVectorizer` usan el mismo analizador. Por defecto:

- pasan el texto a minúsculas (`lowercase=True`);
- extraen tokens con la expresión regular `token_pattern=r'(?u)\b\w\w+\b'`, es decir, **palabras de 2 o más caracteres**.

**Consecuencia importante para el español:** las palabras de una sola letra (*y*, *a*, *o*, *e*, *u*) **se descartan en silencio**. La siguiente celda lo muestra.


In [ ]:
analizador = TfidfVectorizer().build_analyzer()

print(analizador('No me gustó la película'))
print(analizador('Actuaciones brillantes y una trama a medias'))  # 'y' y 'a' desaparecen

> **Interpretación de la salida:**
>
> - Todo queda en minúsculas y la puntuación se elimina.
> - En la segunda frase, *y* y *a* no aparecen. Si necesitas conservarlas (por ejemplo, la conjunción *o* en ciertos análisis), cambia el patrón: `token_pattern=r'(?u)\b\w+\b'`.
> - Las tildes **se conservan** (*gustó*, *película*). Para unificarlas usa `strip_accents='unicode'`.


---

## 3. Paso 1 — TF: frecuencia de cada término en cada documento

Es exactamente la matriz de BoW del notebook anterior.


In [ ]:
cv = CountVectorizer()
tf = cv.fit_transform(corpus).toarray()
terminos = cv.get_feature_names_out()

tf_df = pd.DataFrame(tf, index=doc_ids, columns=terminos)
tf_df

---

## 4. Paso 2 — DF: ¿en cuántos documentos aparece cada término?

`df(t)` cuenta **documentos**, no ocurrencias: si una palabra aparece 3 veces en un mismo documento, ese documento suma 1.


In [ ]:
df_t = (tf > 0).sum(axis=0)

pd.DataFrame({'df(t)': df_t, 'df(t) / N': df_t / N}, index=terminos)

> **Interpretación de la salida:**
>
> - *gusto*, *me* y *pelicula* aparecen en 3 de 5 documentos: son los términos **más comunes** del corpus.
> - *buena*, *la* y *no* aparecen en 2 de 5.
> - Ningún término aparece en un solo documento. Por eso, con unigramas, solo habrá **dos valores distintos de IDF**.


---

## 5. Paso 3 — IDF: penalizar los términos comunes

Calculamos las dos variantes descritas al inicio y las comparamos con el atributo `idf_` que aprende `TfidfVectorizer`.


In [ ]:
idf_libro = np.log10(N / df_t)                   # Jurafsky & Martin
idf_sklearn = np.log((1 + N) / (1 + df_t)) + 1   # scikit-learn (smooth_idf=True)

tfidf_ref = TfidfVectorizer().fit(corpus)        # referencia oficial

idf_tabla = pd.DataFrame(
    {
        'df(t)': df_t,
        'idf libro  log10(N/df)': idf_libro,
        'idf sklearn  ln((1+N)/(1+df))+1': idf_sklearn,
        'TfidfVectorizer.idf_': tfidf_ref.idf_,
    },
    index=terminos,
)
idf_tabla

In [ ]:
print('¿Nuestra fórmula coincide con sklearn?:', np.allclose(idf_sklearn, tfidf_ref.idf_))

# Ejemplo a mano para 'buena' (df = 2):
print(f'ln((1+5)/(1+2)) + 1 = ln(2) + 1 = {np.log(6 / 3) + 1:.4f}')

> **Interpretación de la salida:**
>
> - Las dos variantes **ordenan igual** los términos: menor `df` → mayor IDF. Cambia la escala, no el orden.
> - Con la fórmula de sklearn, *buena*, *la* y *no* (df = 2) obtienen IDF = 1.693, y *gusto*, *me* y *pelicula* (df = 3) obtienen 1.405.
> - Si un término apareciera en **los 5 documentos**, la fórmula del libro le daría IDF = $\log_{10}(5/5) = 0$ (desaparece), mientras que sklearn le daría $\ln(6/6)+1 = 1$ (se conserva con peso mínimo).
> - **Ojo:** el IDF no sabe nada de significado. *no* tiene el mismo IDF que *buena* solo porque ambas aparecen en 2 documentos, no porque sean igual de “informativas”.


---

## 6. Pasos 4 y 5 — Multiplicar TF × IDF y normalizar con L2

### 6.1 TF × IDF sin normalizar


In [ ]:
tfidf_crudo = tf * idf_sklearn   # NumPy multiplica cada columna por su IDF
pd.DataFrame(tfidf_crudo, index=doc_ids, columns=terminos)

### 6.2 Normalización L2

Sin normalizar, un documento largo acumula valores más grandes solo por tener más palabras. La normalización L2 divide cada fila por su longitud euclidiana para que **todos los documentos tengan norma 1**. Así se comparan documentos de distinta longitud en igualdad de condiciones.

**Ejemplo a mano con d1 = “buena pelicula”:**

$$\|d_1\| = \sqrt{1.693^2 + 1.405^2} = \sqrt{2.866 + 1.975} = 2.200$$

$$\text{buena} = 1.693 / 2.200 = 0.769 \qquad \text{pelicula} = 1.405 / 2.200 = 0.639$$


In [ ]:
normas = np.linalg.norm(tfidf_crudo, axis=1, keepdims=True)
tfidf_manual = tfidf_crudo / normas

print('Norma L2 de cada documento antes de normalizar:', normas.ravel().round(3))
pd.DataFrame(tfidf_manual, index=doc_ids, columns=terminos)

### 6.3 Verificación contra `TfidfVectorizer`


In [ ]:
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

print('¿El cálculo manual coincide con TfidfVectorizer?:', np.allclose(tfidf_manual, X_tfidf.toarray()))
print('Normas L2 de las filas de sklearn:', np.linalg.norm(X_tfidf.toarray(), axis=1).round(3))

> **Interpretación de la salida:**
>
> - `True`: los cinco pasos reproducen exactamente `TfidfVectorizer()`.
> - Todas las filas tienen norma 1.0, consecuencia de `norm='l2'`.
> - **d5 = “buena”** tiene un solo término, así que tras normalizar vale **1.0**, el máximo posible.
> - Cuantos más términos tiene un documento, más se reparte el peso: *gusto* vale 0.538 en d3 (3 términos), 0.474 en d4 (4 términos) y 0.412 en d2 (5 términos), aunque aparece una vez en cada uno.
> - En d2 = “no me gusto la pelicula”, *no* y *la* (0.496) pesan más que *me*, *gusto* y *pelicula* (0.412) porque aparecen en menos documentos. Aquí se ve una limitación: TF-IDF le da más peso al artículo *la* que a *gusto*, porque solo mira frecuencias.


### 6.4 Añadir bigramas con `ngram_range=(1, 2)`

Con unigramas, “me gusto” y “no me gusto” comparten casi todos sus términos: la negación queda diluida. Los **bigramas** (pares de palabras consecutivas) capturan expresiones como *no me* o *buena pelicula*.

- `ngram_range=(1, 2)`: incluye unigramas **y** bigramas.
- `min_df=1` (valor por defecto): conserva todo término que aparezca en al menos 1 documento. Con `min_df=2` se eliminarían los términos que aparecen en un único documento.


In [ ]:
tfidf_ng = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_ng = tfidf_ng.fit_transform(corpus)

print(f'Dimensión de la matriz: {X_ng.shape}')
print(f'Términos: {list(tfidf_ng.get_feature_names_out())}')

pd.DataFrame({'df(t)': (X_ng.toarray() > 0).sum(axis=0), 'idf': tfidf_ng.idf_},
             index=tfidf_ng.get_feature_names_out()).sort_values('idf', ascending=False)

In [ ]:
tfidf_ng_df = pd.DataFrame(X_ng.toarray(), index=doc_ids, columns=tfidf_ng.get_feature_names_out())
tfidf_ng_df

> **Interpretación de la salida:**
>
> - El vocabulario pasa de 6 a **11 términos** (6 unigramas + 5 bigramas).
> - *buena pelicula* es el **único término con df = 1** (solo está en d1), así que tiene el IDF más alto (2.099) y es el término de mayor peso en d1 (0.690), por encima de *buena* (0.557).
> - *no me* aparece en d2 y d3: ahora la negación es un rasgo propio que separa “no me gusto” de “me gusto la pelicula”.
> - Costo: los bigramas hacen crecer el vocabulario muy rápido y la matriz se vuelve aún más dispersa.


---

## 7. Comparación lado a lado: BoW vs TF-IDF

Mismo corpus y mismo vocabulario (unigramas). A la izquierda los conteos de BoW; a la derecha los pesos de TF-IDF.


In [ ]:
comparacion = pd.concat(
    {'BoW (conteos)': tf_df,
     'TF-IDF (pesos)': pd.DataFrame(X_tfidf.toarray(), index=doc_ids, columns=terminos)},
    axis=1,
)
comparacion

> **Interpretación de la salida:**
>
> - En BoW, todos los términos presentes valen 1: no hay forma de saber cuál importa más.
> - En TF-IDF, dentro de un mismo documento los términos **menos frecuentes en el corpus** reciben más peso (en d2: *no* y *la* > *gusto*, *me*, *pelicula*).
> - Entre documentos, el mismo término pesa distinto según la longitud del documento (*gusto*: 0.412 en d2, 0.538 en d3, 0.474 en d4).
> - TF-IDF **no cambia qué celdas son cero**: la dispersión es la misma que en BoW.


---

## 8. Similitud coseno entre documentos

La **similitud coseno** mide el ángulo entre dos vectores (Jurafsky & Martin, §6.4):

$$\cos(\vec{a}, \vec{b}) = \frac{\vec{a}\cdot\vec{b}}{\|\vec{a}\|\,\|\vec{b}\|}$$

Vale 1 si los documentos tienen exactamente la misma distribución de pesos y 0 si no comparten ningún término (con pesos no negativos nunca es negativa). Como `TfidfVectorizer` ya deja cada fila con norma 1, el coseno se reduce al **producto punto**.


In [ ]:
sim_bow = cosine_similarity(tf)
sim_tfidf = cosine_similarity(X_tfidf)

print('¿Coseno = producto punto tras normalizar L2?:',
      np.allclose(sim_tfidf, (X_tfidf @ X_tfidf.T).toarray()))

pd.concat(
    {'Coseno con BoW': pd.DataFrame(sim_bow, index=doc_ids, columns=doc_ids),
     'Coseno con TF-IDF': pd.DataFrame(sim_tfidf, index=doc_ids, columns=doc_ids)},
    axis=1,
)

> **Interpretación de la salida:**
>
> - La diagonal vale 1: cada documento es idéntico a sí mismo.
> - **d2 y d4** (“no me gusto la pelicula” y “me gusto la pelicula”) son el par más parecido: 0.894 con BoW y 0.868 con TF-IDF, aunque **expresan opiniones opuestas**. Solo los separa *no*, y con unigramas ni BoW ni TF-IDF capturan la negación. Es una limitación clave para el análisis de sentimiento.
> - **d1 y d5** (“buena pelicula” y “buena”) suben de 0.707 con BoW a 0.769 con TF-IDF: comparten *buena*, que tiene más IDF que *pelicula*.
> - TF-IDF baja la similitud cuando lo compartido son términos comunes: d3–d4 pasa de 0.577 a 0.510 porque solo comparten *me* y *gusto* (df = 3).
> - d1 y d3 no comparten términos → similitud 0.


---

# Parte B — Corpus real: `dataset_reseñas_espanol.csv`

## 9. Cargar el conjunto de datos y detectar duplicados

Es el mismo conjunto de `bow_model.ipynb`: 200 reseñas de películas en español etiquetadas como *Positiva* / *Negativa*. Antes de calcular IDF hay que revisar cuántas reseñas son **distintas**, porque el DF cuenta documentos y cada copia de una reseña cuenta como un documento más.


In [ ]:
reseñas = pd.read_csv(DIR_DATA / 'dataset_reseñas_espanol.csv')

print(f'Filas: {len(reseñas)}')
print(f'Reseñas distintas: {reseñas["Reseña"].nunique()}')
print('Distribución de clases:', reseñas['Sentimiento'].value_counts().to_dict())
reseñas.head()

In [ ]:
repeticiones = reseñas.value_counts(['Reseña', 'Sentimiento']).rename('repeticiones').reset_index()
repeticiones

> **Interpretación de la salida:**
>
> - Las 200 filas corresponden a solo **20 reseñas distintas** (10 positivas y 10 negativas), repetidas entre 5 y 16 veces cada una.
> - Las clases están balanceadas (100/100), pero las repeticiones no son uniformes.
> - Para TF-IDF esto importa: una palabra que solo aparece en una reseña repetida 16 veces tendrá `df = 16`, aunque en realidad pertenezca a **un único texto**. La siguiente sección mide ese efecto.
> - Para entrenar un clasificador, los duplicados también provocan **fuga de datos** (*data leakage*): la misma reseña puede quedar en entrenamiento y en prueba.


---

## 10. TF-IDF con y sin duplicados

Ajustamos `TfidfVectorizer` dos veces: con las 200 filas y con las 20 reseñas únicas. Comparamos el IDF de cada término.


In [ ]:
unicas = reseñas.drop_duplicates(subset='Reseña').reset_index(drop=True)

tfidf_200 = TfidfVectorizer()
X_200 = tfidf_200.fit_transform(reseñas['Reseña'])

tfidf_20 = TfidfVectorizer()
X_20 = tfidf_20.fit_transform(unicas['Reseña'])

print(f'Matriz con duplicados: {X_200.shape}   dispersión: {1 - X_200.nnz / np.prod(X_200.shape):.2%}')
print(f'Matriz sin duplicados: {X_20.shape}    dispersión: {1 - X_20.nnz / np.prod(X_20.shape):.2%}')

In [ ]:
def df_de(X):
    return np.asarray((X > 0).sum(axis=0)).ravel()

idf_comp = (
    pd.DataFrame({'df (200)': df_de(X_200), 'idf (200)': tfidf_200.idf_},
                 index=tfidf_200.get_feature_names_out())
    .join(pd.DataFrame({'df (20)': df_de(X_20), 'idf (20)': tfidf_20.idf_},
                       index=tfidf_20.get_feature_names_out()))
)

ejemplos = ['muy', 'película', 'actuaciones', 'no', 'historia', 'excelente', 'conmovedora', 'rescatable']
idf_comp.loc[ejemplos]

> **Interpretación de la salida:**
>
> - Ambas matrices tienen el **mismo vocabulario (84 términos)**: los duplicados no añaden palabras nuevas, solo cambian los DF.
> - **Sin duplicados**, todas las palabras que aparecen en una sola reseña (*excelente*, *conmovedora*, *rescatable*…) reciben el mismo IDF (3.351), lo cual es correcto: son igual de exclusivas.
> - **Con duplicados**, esas mismas palabras reciben IDF distintos (*conmovedora* 4.512, *rescatable* 4.224, *excelente* 4.001) solo porque su reseña se repite 5, 7 o 9 veces. El IDF está midiendo **cuántas copias hay**, no qué tan distintiva es la palabra.
> - *actuaciones* aparece en 3 reseñas distintas. Sin duplicados su IDF (2.658) es menor que el de una palabra exclusiva, como debe ser. Con duplicados, su DF sube a 25 filas.
> - **Conclusión:** para calcular IDF conviene eliminar duplicados exactos antes de `fit`. Las secciones siguientes usan las **20 reseñas únicas**.


---

## 11. Términos clave

### 11.1 Términos más comunes y más distintivos del corpus (según IDF)


In [ ]:
idf_20 = pd.Series(tfidf_20.idf_, index=tfidf_20.get_feature_names_out(), name='idf')

print('Términos MÁS comunes (IDF más bajo):')
print(idf_20.sort_values().head(10).round(3).to_string())
print(f'\nTérminos con el IDF máximo ({idf_20.max():.3f}): {(idf_20 == idf_20.max()).sum()} de {len(idf_20)}')

> **Interpretación de la salida:**
>
> - Los términos más comunes son palabras funcionales (*muy*, *una*, *un*, *la*, *los*) y palabras del dominio (*película*, *actuaciones*, *historia*, *guion*): aparecen en varias reseñas sin importar si son positivas o negativas.
> - **69 de los 84 términos** aparecen en una sola reseña y comparten el IDF máximo (3.351). En un corpus tan pequeño, el IDF casi no distingue entre ellos.
> - El IDF no usa la etiqueta de sentimiento: un término “distintivo” no es necesariamente útil para clasificar.


### 11.2 Términos de mayor peso en cada reseña

Para cada reseña tomamos los 3 términos con mayor TF-IDF. Es una forma simple de extraer **palabras clave** de un documento.


In [ ]:
def top_terminos(X, nombres, k=3):
    X = X.toarray()
    salida = []
    for fila in X:
        idx = fila.argsort()[::-1][:k]
        salida.append(', '.join(f'{nombres[i]} ({fila[i]:.2f})' for i in idx if fila[i] > 0))
    return salida

nombres_20 = tfidf_20.get_feature_names_out()
pd.set_option('display.max_colwidth', None)
pd.DataFrame({
    'Reseña': unicas['Reseña'],
    'Sentimiento': unicas['Sentimiento'],
    'Top-3 términos TF-IDF': top_terminos(X_20, nombres_20),
})

> **Interpretación de la salida:**
>
> - En muchas reseñas los términos principales tienen carga de opinión (*pésimas*, *aburrida*, *decepcionante*, *entretenida*): aparecen en una sola reseña y tienen el IDF máximo.
> - **Empates:** como 69 términos comparten el mismo IDF, en la mayoría de las reseñas los tres primeros tienen exactamente el mismo peso, y su orden es arbitrario. Conviene mirar el valor y no solo la posición.
> - **Palabras funcionales con peso alto:** *con* (reseña 3), *mi* (reseña 9) o *son* (reseña 12) aparecen en el top-3 al mismo nivel que palabras de opinión, solo porque en este corpus aparecen en una única reseña. El IDF no sabe qué es una *stopword*. Para eliminarlas hay que pasar una lista con `stop_words=[...]` (sklearn solo trae la lista en inglés).
> - Las palabras más comunes (*muy*, *una*) solo llegan al top-3 en reseñas muy cortas, como “Muy entretenida y emotiva.”.


---

## 12. Similitud coseno entre reseñas reales

Calculamos la similitud entre las 20 reseñas únicas y listamos los 5 pares más parecidos.


In [ ]:
sim_20 = cosine_similarity(X_20)

i, j = np.triu_indices_from(sim_20, k=1)   # cada par una sola vez, sin la diagonal
pares = pd.DataFrame({
    'Reseña A': unicas['Reseña'].values[i],
    'Sent. A': unicas['Sentimiento'].values[i],
    'Reseña B': unicas['Reseña'].values[j],
    'Sent. B': unicas['Sentimiento'].values[j],
    'coseno': sim_20[i, j],
}).sort_values('coseno', ascending=False)

print(f'Pares sin ningún término en común: {(pares["coseno"] == 0).sum()} de {len(pares)}')
pares.head(5)

> **Interpretación de la salida:**
>
> - **145 de los 190 pares** (76 %) tienen similitud **0**: con reseñas tan cortas casi no hay vocabulario compartido.
> - El par **más parecido** (0.305) es “Gran dirección y un guion muy sólido.” (positiva) frente a “Un guion pobre y sin emoción.” (negativa). Comparten *un* y *guion*, pero **expresan opiniones opuestas**.
> - El segundo par (0.294) también mezcla sentimientos: “Una trama aburrida y predecible.” frente a “Actuaciones brillantes y una trama muy bien lograda.”, unidas por *una* y *trama*.
> - Los pares se parecen por el **tema** (guion, trama, actuaciones), no por la opinión.
> - Esto confirma la limitación vista en la Parte A: TF-IDF mide **solapamiento léxico**, no significado. Sinónimos como *aburrida* y *tediosa* tendrían similitud 0.


---

## Resumen

| Aspecto | Bag-of-Words | TF-IDF |
|---------|--------------|--------|
| Representación | Conteos brutos | Conteos ponderados por IDF (y normalizados L2 en sklearn) |
| Palabras comunes | Mismo peso que las demás | Peso reducido por el IDF |
| Palabras distintivas | Mismo peso que las demás | Peso aumentado |
| Longitud del documento | Los documentos largos tienen vectores más grandes | Se compensa con `norm='l2'` |
| Orden de las palabras | Se pierde | Se pierde (los n-gramas lo recuperan en parte) |
| Dispersión | Alta | Igual de alta |
| Semántica (sinónimos, negación) | No la captura | No la captura |

### Puntos clave

1. TF-IDF = **TF × IDF**, y en scikit-learn además se aplica **normalización L2**. La fórmula del libro y la de sklearn ordenan igual los términos, pero dan valores distintos.
2. El IDF solo depende de **en cuántos documentos** aparece un término, no de su significado ni de la etiqueta.
3. Los **duplicados** inflan el DF y distorsionan el IDF: elimínalos antes de ajustar el vectorizador.
4. Los **bigramas** ayudan a capturar expresiones y negaciones (*no me*), a costa de un vocabulario mayor.
5. La **similitud coseno** con TF-IDF mide solapamiento de palabras. Dos reseñas con opiniones opuestas pueden ser muy “similares”.

### Siguiente paso

Usar la matriz TF-IDF como entrada de un clasificador → ver `tfidf_naive_bayes_imdb.ipynb`. Para capturar similitud semántica (sinónimos), ver *word embeddings* en `text_representation.ipynb`.
